# Part 1 - Introduction to Catalyst

This notebook provides a basic introduction on how to use the [Catalyst.jl](https://github.com/SciML/Catalyst.jl) package's `@reaction_network` DSL to simulate chemical reaction network models.


## Part 1.1 - Basic DSL simulation workflow

Catalyst implements a so-called *domain-specific language* for *chemical reaction network* modelling. This is a specific type of model commonly used in biology and chemistry. Catalyst permits implementing this using an intuitive notation. Non-reaction network models will be described later on in the workshop.

Here we declare a simple two-state process where a a species flips between two forms at constant rates.
- The model is stored in the variable `two-state_model`.
- The model declaration is enclosed by the `@reaction_network begin ... end` block.
- Each line is one "reaction event". First we write the reaction rate, followed by the reaction itself.
- We also need to import the Catalyst package with `using Catalyst`.

In [ ]:
using Catalyst
two_state_model = @reaction_network begin
    k1, X1 --> X2
    k2, X2 --> X1
end


SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice; giving up


To simulate the model we will first couple the model, an initial condition, parameter values, and a simulation time end point into a so-called `ODEProblem`. We can then simulate the model using `solve` (which requires the [OrdinaryDiffEq.jl](https://github.com/SciML/OrdinaryDiffEq.jl) package).

In [ ]:
using OrdinaryDiffEq
u0 = [:X1 => 2.0, :X2 => 1.0] 
tend = 10.0
ps = [:k1 => 1.0, :k2 => 0.2]
ode_prob = ODEProblem(two_state_model, u0, tend, ps)
ode_sol = solve(ode_prob); # Here, `;` is used to suppress output printing in the notebook.

We can now plot the simulation using the `plot` command from the [Plots.jl](https://github.com/JuliaPlots/Plots.jl) package.

In [ ]:
using Plots
plot(ode_sol)

## Part 1.2 - Available simulation types

Above we used an ODE simulation. Here, ODEs are generated from a chemical reaction network model using the (to the law of mass-action related) *Reaction Rate Equation*. To see the ODEs that we simulate we use the `ode_model` function:

In [ ]:
ode_model(two_state_model)

Alternatively, Catalyst can simulate the model as a stochastic differential equation (generated through the *chemical Langevin equation*) or as a jump process (generated through *stochastic chemical kinetics*, although the Gillespie algorithm with which this is simulated is often the most well-known term).

To make a SDE simulation, we simple use the [StochasticDiffEq.jl](https://github.com/SciML/StochasticDiffEq.jl) package and create an `SDEProblem` instead. Otherwise, the syntax is identical.

In [ ]:
using StochasticDiffEq
sde_prob = SDEProblem(two_state_model, u0, tspan, ps)
sde_sol = solve(sde_prob)
plot(sde_sol)

For jump simulations, we use the [JumpProcesses.jl](https://github.com/SciML/JumpProcesses.jl) package and go through a `JumpProblem`. 

In [ ]:
using JumpProcesses
jump_prob = JumpProblem(two_state_model, u0, tspan, ps)
jump_sol = solve(jump_prob)
plot(jump_sol)

## Part 1.3 - Options for creating reactions in the DSL 

Next we will consider different approaches for creating reactions using the DSL.

#### Various reactions types

Sometimes, some species are produced (or degraded). This can be represented through the empty set (using either `0` or `∅`). Here we create a simple production/degradation network:

In [ ]:
rn_2 = @reaction_network begin
    p, 0 --> X    # X is produced.
    d, X --> 0    # X is degraded.
end

A reaction's substrates and/or products can contain multiple species. These are separated by a `+`. Here we create a simple binding/unbinding network (where `X` and `Y` reversibly bind to form `XY`).

In [ ]:
rn_3 = @reaction_network begin
    kB, X + Y --> XY     # X and Y binds to form XY.
    kD, XY --> X + Y     # XY dissociates into X and Y.
end

If multiple copies of a species participate in a reaction, the species can be preceded by a number indicating their stoichiometry. Here, we model a (reversible) dimerisation reaction (where 2 `X` molecules form a single molecule `X2`).

In [ ]:
rn_4 = @reaction_network begin
    kB, 2X --> X2     # X dimerises to form X2.
    kD, X2 --> 2X     # X2 dissociates into 2 copies of X.
end

These features can be combined freely, and stoichiometries can even be nested.

In [ ]:
wacky_reaction = @reaction_network begin
    kW, 2X + 3(Y +2Z) --> 5V + W    
end

#### Bundling of similar reactions
Often, reaction networks contain reactions with similar structures. These can be bundled to create a more concise notation.

Reversible reactions can be modelled using a `<-->`. Here, the rate becomes a *tuple*, `(.,.)`, describing the *forward rate* followed by the *backward rate*. Here we create a production/degradation system in a single like:

In [ ]:
rn_5 = @reaction_network begin
    (p,d), 0 <--> X     
end

This is identical to the notation we used previously (to create `rn_2`):

In [ ]:
rn_5 = @reaction_network begin
    p, 0 --> X    # X is produced.
    d, X --> 0    # X is degraded.
end

Tuples can also be applied to the substrate/product expressions to bundle similar terms. If we have a system of two species `X` and `Y` degrading at rates `dX` and `dY`, these reactions can be bundled accordingly:

In [ ]:
rn_6 = @reaction_network begin
    (dX,dY), (X,Y) --> 0
end

If we wish both reactions to use the same rate `d`, this is possible:

In [ ]:
rn_7 = @reaction_network begin
    d, (X,Y) --> 0
end

If we wish, bundling can be used for both substrates and products. Here, we create two reactions one where `X1` is converted into `X2` (at rate `kX`) and one where `Y1` is converted into `Y2` (at rate `kY`).

In [ ]:
rn_8 = @reaction_network begin
    (kX,kY), (X1,Y1) --> (X2,Y2)
end

Again, bundling can be combined in various ways, using any number of terms, and also be combined with reversible reactions:

In [ ]:
wacky_network = @reaction_network begin
    ((pX, pY, pZ),d), (0, Y0, Z0) <--> (X, Y, Z1+Z2)
end

The important rule is that the rate is first split into the forward and backward reactions, and bundling is then applied (in this case, bundling has probably been applied overzealously, making it harder to figure out what is going on).

#### Using special symbols
Julia supports all Unicode characters, and these can be used in reaction networks.

The empty set can be represented through the `∅` symbol, and most arrows (e.g. `→`, `↣`, `↔`, `⇄`, etc.) can be used. E.g here we recreate our production/degradation reaction using a fancier notation.

In [ ]:
rn_9 = @reaction_network begin
    (p,d), ∅ ↔ X 
end

Such characters can also be used for species and parameters. E.g. when I worked on the bacterial sigma factor V (*σᵛ*), I could create a reaction where it is produced at the basic rate *v₀* using:



In [ ]:
rn_10 = @reaction_network begin
    v₀, ∅ → σᵛ
end

If you want, backwards arrows can also be used:

In [ ]:
rn_11 = @reaction_network begin
    d, 0 <-- X    # Equivalent to X --> 0
end

Also possible:

In [ ]:
rn_12 = @reaction_network begin
    🍦, 😢 --> 😃
end

#### Non-constant reaction rates
So far we have assumed that reaction rates are all constant parameters, however, most expressions are permitted in the rate terms.

Let's assume that the activation of species `X` (converting it from an inactive form `X_i` to an active form `X_a`) is catalysed by the enzyme `E`. We can here put `E` in the reaction rate. In this case, we will also add model production/degradation of `E`.

In [ ]:
rn_13 = @reaction_network begin
    (p,d), 0 <--> E
    E, X_i --> X_a
end

When can now check the rate of change in `X_a` in the corresponding ODE using `ode_model`:

In [ ]:
ode_model(rn_13)

If we want to also make the rate of the reaction `X_i --> X_a` scale with a parameter `k`, we can add that to the rate:

In [ ]:
rn_14 = @reaction_network begin
    (p,d), 0 <--> E
    k*E, X_i --> X_a
end

In this case, an equivalent model can be achieved by adding `E` as a substrate and product to the reaction it catalyses:

In [ ]:
rn_14 = @reaction_network begin
    (p,d), 0 <--> E
    k, X_i + E --> X_a + E
end

(in fact, if one carries out jump simulations this as advantageous, as their speed can be increased if all reaction rates are constant)

The Michaelis-Menten (*mm(X,v,K) = v\*X/(X + K)*) and Hill (*hill(X,v,K,n) = v\*X^n/(X^n + K^n)*) functions are frequently used in systems biology. These are natively supported. Here we create a model where the rate of production of `X` is determined by a Hill function according to the concentration of a transcription factor (`T`, again we let `T` be produced and degraded at a constant rate).

In [ ]:
rn_15 = @reaction_network begin
    (p,d), 0 <--> T
    hill(T,v,K,n), 0 --> X
end

A few other functions, such as repressive Michaelis-Menten and Hill functions are supported. It is also possible for the user to define their own functions and use them within the DSL.

In [ ]:
my_function(k1,k2,E) = (k1+E)/(k2+E)
rn_16 = @reaction_network begin
    (p,d), 0 <--> E
    my_function(K1,K2,E), X_i --> X_a  # The name of the inputs to custom functions does not need to be the same as used to declare it.
end

Generally, any valid expression can be used, and we can also use any normal Julia functions:

In [ ]:
rn_17 = @reaction_network begin
    (p,d), 0 <--> E
    log(E) + k1^(1+k2), X_i --> X_a  
end

The symbol `t` is reserved for time, and can be used to create time-dependent reactions. Here, `X` is produced at a cyclic rate (possibly modelling a circadian rythm):

In [ ]:
rn_18 = @reaction_network begin
   A*(1 + sin(2π/T*t+ϕ))/2, 0 --> X
end

Finally, reaction rates can also be constants. Here `X` is produced at the constant rate `1`.

In [ ]:
rn_19 = @reaction_network begin
    1, 0 --> X
 end

## Part 1.4 - Examples of reaction network models

While the chemical reaction network notation to used in Catalyst to create models might seem niche, it actually encompasses a very large range of models, some of which we will exemplify here. Furthermore, as we will describe later, non-reaction network components can also be integrated into the model

#### Systems Biology Example: The repressilator

In [ ]:
repressilator = @reaction_network begin
    hillr(Z,v,K,n), ∅ --> X
    hillr(X,v,K,n), ∅ --> Y
    hillr(Y,v,K,n), ∅ --> Z
    d, (X, Y, Z) --> ∅
end

u0 = [:X => 50.0, :Y => 15.0, :Z => 15.0]
ps = [:v => 10.0, :K => 20.0, :n => 3, :d => 0.1]
oprob = ODEProblem(repressilator, u0, 200.0, ps)
osol  = solve(oprob)
plot(osol)

#### Chemistry Example: Cross-coupling

In [ ]:
cc_system = @reaction_network begin
    k₁, S₁ + C --> S₁C
    k₂, S₁C + S₂ --> CP
    k₃, CP --> C + P
end

u0 = [:S₁ => 1.0, :C => 0.05, :S₂ => 1.2, :S₁C => 0.0, :CP => 0.0, :P => 0.0]
ps = [:k₁ => 5.0, :k₂ => 5.0, :k₃ => 100.0]
oprob = ODEProblem(cc_system, u0, 25.0, ps)
osol  = solve(oprob)
plot(osol)

#### Epidemiology Example: The SIR model

In [ ]:
sir_model = @reaction_network begin
    α, S + I --> 2I
    β, I --> R
end

u0 = [:S => 99, :I => 1, :R => 0]
ps = [:α => 0.001, :β => 0.01]
oprob = ODEProblem(sir_model, u0, 500.0, ps)
osol = solve(oprob)
plot(osol)

#### Population dynamics: Logistic growth

In [ ]:
log_growth = @reaction_network begin
    r, X --> 2X
    r/K, 2X --> X
end

u0 = [:X => 0.5]
ps = [:r => 2.0, :K => 4.0]
oprob = ODEProblem(log_growth, u0, 5.0, ps)
osol = solve(oprob)
plot(osol)

#### Pharmacology Example: Drug delivery

In [ ]:
drug_delivery = @reaction_network begin
    (k_bl,k_lb), D_blood <--> D_liver
    k_abs, D_liver --> D_abs
    d_l, D_liver --> 0
    d_b, D_blood --> 0
end

u0 = [:D_blood => 1.0, :D_liver => 0.0, :D_abs => 0.0]
ps = [:k_bl => 0.2, :k_lb => 0.3, :k_abs => 2.0, :d_l => 0.2, :d_b => 0.25]
oprob = ODEProblem(drug_delivery, u0, 10.0, ps)
osol = solve(oprob)
plot(osol)

## Part 1.5 - Working with solutions
Next, we will consider some options for working with simulations. Let us first simulate a Brusselator ODE model.

In [ ]:
# Declare model.
brusselator = @reaction_network begin
    A, ∅ → X
    1, 2X + Y → 3X
    B, X → Y
    1, X → ∅
end

# Simulate the model for a specific condition.
u0 = [:X => 1.0, :Y => 0.0]
ps = [:A => 1.0, :B => 4.0]
oprob = ODEProblem(brusselator, u0, 50.0, ps)
sol = solve(oprob)
plot(sol)

If we only want to plot a specific species, we can use the `idxs` option to `plot`. Here we plot `X` only:

In [ ]:
plot(sol; idxs = :X)

Phase-space plots can also be achieved through the `idxs` argument.

In [ ]:
plot(sol; idxs = (:X, :Y))

Specifically, options such `linewidth`, `color`, `xguide`, etc. can be used to customise the plot.

In [ ]:
plot(sol; idxs = :Y, linewidth = 4, color = :red, xguide = "Time")

To access the values in a simulation we can index it with a specific variable, i.e. to get all `X` values we use:

In [ ]:
sol[:X]

We can also sample it at specific time points using `()` brackets. Next, the solution will be automatically interpolated at the desiganted time points. We then use `idxs` to designate the species we are interested in.

In [ ]:
sol(1:2:50; idxs = :X)

## Part 1.6 - More on simulations

#### Simulation solver options
The `solve` command takes various arguments to e.g. specify solver adaptivity, time step sizing, and similar. It also allows you to 
specify a solver algorithm. When an argument is not given, default values are automatically determined.

To e.g. specify that we wish to simulate the Brusselator using the stiff `Rodas5P` solver, we use:

In [ ]:
solve(oprob, Rodas5P());

To e.g. turn off adaptive time stepping and set a fixed dt, use:

In [ ]:
solve(oprob, Rodas5P(); adaptive = false, dt = 0.01);

#### Non-standard time spans

So-far we have just provided a single simulation final time point. However, we can also provide a full timespan:

In [ ]:
# Declare model.
brusselator = @reaction_network begin
    A, ∅ → X
    1, 2X + Y → 3X
    B, X → Y
    1, X → ∅
end

# Simulate the model for a specific condition.
u0 = [:X => 1.0, :Y => 0.0]
tspan = (0.0, 50.0)
ps = [:A => 1.0, :B => 4.0]
oprob = ODEProblem(brusselator, u0, tspan, ps)
sol = solve(oprob)
plot(sol)

However, the time span does not have to start at 0, it can start anywhere:

In [ ]:
tspan = (20.0, 50.0)
oprob = ODEProblem(brusselator, u0, tspan, ps)
sol = solve(oprob)
plot(sol)

It can also start at negative values

In [ ]:
tspan = (-20.0, 50.0)
oprob = ODEProblem(brusselator, u0, tspan, ps)
sol = solve(oprob)
plot(sol)

And if the first value is before the second value, we will simulate backwards in time:

In [ ]:
tspan = (50.0, 49.8) # The brusselator turns quickly unstable and goes towards infinity when you simulate backwards. generally, while this is possible, expect weird results.
oprob = ODEProblem(brusselator, u0, tspan, ps)
sol = solve(oprob)
plot(sol)

#### Monte Carlo simulations
In certain situations, we want to carry out multiple simulations simultaneously. Examples include making repeat simulations of a stochastic model, or simulating a model with several different initial conditions or parameter values. We can do this by creating an `EnsembleProblem`, which can then be simulated using various forms of parallelisation.

Let us consider a simple bistable switch model, and consider how the stochastic system randomly switches from an inactive state (where we set the initial condition) to an active state. First, we create a normal `SDEProblem`.

In [ ]:
using StochasticDiffEq
bistable_switch = @reaction_network begin
    v0 + hill(X,v,K,n), ∅ --> X
    deg, X --> ∅
end
u0 = [:X => 0.0]
tspan = (0.0, 1000.0)
p = [:v0 => 0.1, :v => 2.5, :K => 75.0, :n => 2.0, :deg => 0.01]
sprob = SDEProblem(bistable_switch, u0, tspan, p);

Next, we supply our `SDEProblem` to an `EnsembleProblem`. Here, since we wish to simulate the same SDE repeatedly, it is the only input to the `EnsembleProblem`. However, if we e.g. would like to use different initial conditions for every simulation, this kind of information is also supplied here.

In [ ]:
eprob = EnsembleProblem(sprob);

Finally, we can perform Monte Carlo simulations with the `solve` command, using the `trajectories` argument to designate how many simulations we want. We can plot the output using the `plot` command.

In [ ]:
esol = solve(eprob; trajectories = 100)
plot(esol)

Sometimes, when plotting multiple Monte Carlo trajectories, the `linealpha` argument (which makes trajectories more transparent) can be useful.

In [ ]:
plot(esol; linealpha = 0.4)

When simulating an `EnsembleProblem`, an ensemble algorithm can be chosen (e.g. `EnsembleSerial()`, `EnsembleThreads()`, or `EnsembleGPUArray()`) to determine parallelisation strategy. By default `EnsembleThreads()` is used (best for problems where each individual simulation is quick).

Here, we use `EnsembleDistributed()` to parallelise over several processors.

In [ ]:
esol = solve(eprob, ImplicitEM(), EnsembleDistributed(); trajectories = 100)
plot(esol; linealpha = 0.5)